# First Getting Data from Kraken descriptors
The first step is to get the smiles from smiles.csv and get the required data from autoqchem

In [ ]:
from rdkit import Chem
import pandas as pd

def canonicalise(smiles:str)->str:
    """
    Given a SMILES string, return its canonical form.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
       print('Invalid SMILES:', smiles)
       return ''

    return Chem.MolToSmiles(mol, canonical=True)

def generate_inchikey(smiles:str)->str:
    """
    Given a SMILES string, return its InChIKey.
    """
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
       print('Invalid SMILES:', smiles)
       return ''

    return Chem.inchi.MolToInchiKey(mol)

my_subset_path = "data/Ligands_pics/smiles.csv"
descriptor_path = "data/descriptors_DFT.csv"

# Read the CSV file into a DataFrame
df_subset = pd.read_csv(my_subset_path, header=0)

df_all = pd.read_csv(descriptor_path, header=0)

df_subset['canonical_smiles'] = df_subset['smiles'].apply(canonicalise)
df_subset['inchikey'] = df_subset['smiles'].apply(generate_inchikey)
df_all['canonical_smiles'] = df_all['smiles'].apply(canonicalise)
df_all['inchikey'] = df_all['smiles'].apply(generate_inchikey)

subset_inchikeys = set(df_subset['inchikey'].unique())
set_of_all_inchikeys = set(df_all['inchikey'].unique())

# Find the intersection of the two sets
intersection_inchikeys = subset_inchikeys.intersection(set_of_all_inchikeys)
print(f"Number of common InChIKeys: {len(intersection_inchikeys)}")
print("Common InChIKeys:")
for inchikey in intersection_inchikeys:
    print(inchikey)

# do the same with smiles:
subset_smiles = set(df_subset['canonical_smiles'].unique())
set_of_all_smiles = set(df_all['canonical_smiles'].unique())
# Find the intersection of the two sets
intersection_smiles = subset_smiles.intersection(set_of_all_smiles)
print(f"Number of common SMILES: {len(intersection_smiles)}")
print("Common SMILES:")
for smiles in intersection_smiles:
    print(smiles)

# print the name column of the common inchi and common smiles from the subset:
print("Common InChIKeys and their names:")
for inchikey in intersection_inchikeys:
    name = df_subset[df_subset['inchikey'] == inchikey]['name'].values[0]
    print(f"InChIKey: {inchikey}, Name: {name}")

print("Common SMILES and their names:")
for smiles in intersection_smiles:
    name = df_subset[df_subset['canonical_smiles'] == smiles]['name'].values[0]
    print(f"SMILES: {smiles}, Name: {name}")


# all seems in agreement. make a DF with our 9 smiles and all their descriptor

smiles_with_descriptors = pd.merge(df_subset, df_all, on='inchikey', how='inner')
print(smiles_with_descriptors.shape)
# smiles_with_descriptors.to_csv("data/smiles_with_descriptors.csv", index=False)


# Getting PCA of the descriptors

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.manifold import TSNE, Isomap
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
import numpy as np
import umap


# Select the columns to be used for PCA
#
labels_columns = ['name','smiles_x','cas','canonical_smiles_x','inchikey','ID','smiles_y', 'canonical_smiles_y','inchikey_y']

descriptor_columns = [col for col in smiles_with_descriptors.columns if col not in labels_columns]

print(descriptor_columns)
# Standardize the descriptor data and extract labels
X = smiles_with_descriptors[descriptor_columns].dropna()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
labels = smiles_with_descriptors['name']

def reduce_dimensions(X, method='PCA', n_components=2, labels=None):
    """
    Perform dimensionality reduction using a specified method.

    Parameters:
        X (array-like): The standardized data.
        method (str): One of 'PCA', 'TSNE', 'UMAP', 'LDA', or 'Isomap'.
        n_components (int): Number of dimensions to reduce to.
        labels (array-like): Required for LDA.

    Returns:
        X_reduced (ndarray): The reduced data.
    """
    if method == 'PCA':
        reducer = PCA(n_components=n_components)
        X_reduced = reducer.fit_transform(X)
    elif method == 'TSNE':
        reducer = TSNE(n_components=n_components, random_state=42, perplexity=2)
        X_reduced = reducer.fit_transform(X)
    elif method == 'UMAP':
        reducer = umap.UMAP(n_components=n_components, random_state=42)
        X_reduced = reducer.fit_transform(X)
    elif method == 'LDA':
        if labels is None:
            raise ValueError("Labels must be provided for LDA.")
        reducer = LDA(n_components=n_components)
        X_reduced = reducer.fit_transform(X, labels)
    elif method == 'Isomap':
        reducer = Isomap(n_components=n_components)
        X_reduced = reducer.fit_transform(X)
    else:
        raise ValueError("Method not recognized. Choose from 'PCA', 'TSNE', 'UMAP', 'LDA', or 'Isomap'.")
    return X_reduced

# ---- Change these parameters as desired ----
method = 'UMAP'        # Options: 'PCA', 'TSNE', 'UMAP', 'LDA', 'Isomap'
n_components = 2      # Set to 1 for 1D reduction, 2 for 2D, etc.
# ---------------------------------------------

# Perform the dimensionality reduction
X_reduced = reduce_dimensions(X_scaled, method=method, n_components=n_components, labels=labels)

# Create a DataFrame to hold the reduced dimensions and the label for plotting
dim_names = [f"Dim{i+1}" for i in range(n_components)]
df_reduced = pd.DataFrame(data=X_reduced, columns=dim_names)
df_reduced['name'] = labels.values

# Normalize dimensions for a consistent plot (optional)
for col in dim_names:
    df_reduced[col] = (df_reduced[col] - df_reduced[col].min()) / (df_reduced[col].max() - df_reduced[col].min())

# Plotting the reduced dimensions
plt.figure(figsize=(6, 6))
if n_components == 1:
    # 1D plot: all points along a line
    plt.scatter(df_reduced['Dim1'], np.zeros_like(df_reduced['Dim1']), alpha=0.5)
    for i, txt in enumerate(df_reduced['name']):
        plt.annotate(txt, (df_reduced['Dim1'][i], 0), fontsize=8)
    plt.xlabel("Dim1")
else:
    # 2D plot
    plt.scatter(df_reduced['Dim1'], df_reduced['Dim2'], alpha=0.5)
    for i, txt in enumerate(df_reduced['name']):
        plt.annotate(txt, (df_reduced['Dim1'][i], df_reduced['Dim2'][i]), fontsize=8)
    plt.xlabel("Dim1")
    plt.ylabel("Dim2")
plt.title(f"{method} Dimensionality Reduction (n_components={n_components})")
plt.show()

In [ ]:
# do the same on the whole dataset and see how the PCA loading of the things we want compare:
# Select the columns to be used for PCA
columns_to_remove = ['ID', 'smiles', 'canonical_smiles', 'inchikey']

descriptor_columns_all = [col for col in df_all.columns if col not in columns_to_remove]
df_all = df_all.dropna(subset=descriptor_columns_all)
# Standardize the descriptor data and extract labels
X_all = df_all[descriptor_columns_all].dropna()
scaler_all = StandardScaler()
X_scaled_all = scaler_all.fit_transform(X_all)
labels_all = df_all['canonical_smiles']

# Perform the dimensionality reduction
X_reduced_all = reduce_dimensions(X_scaled_all, method=method, n_components=n_components, labels=labels_all)

# Create a DataFrame to hold the reduced dimensions and the label for plotting
dim_names_all = [f"Dim{i+1}" for i in range(n_components)]
df_reduced_all = pd.DataFrame(data=X_reduced_all, columns=dim_names_all)
df_reduced_all['canonical_smiles'] = labels_all.values

# Normalize dimensions for a consistent plot (optional)
for col in dim_names_all:
    df_reduced_all[col] = (df_reduced_all[col] - df_reduced_all[col].min()) / (df_reduced_all[col].max() - df_reduced_all[col].min())
# Plotting the reduced dimensions


# get the subset of things we want:
reduced_with_names = pd.merge(df_reduced_all, smiles_with_descriptors ,left_on='canonical_smiles', right_on = 'canonical_smiles_x', how='inner')
# normalise dimensions for a consistent plot (optional)
for col in dim_names_all:
    reduced_with_names[col] = (reduced_with_names[col] - reduced_with_names[col].min()) / (reduced_with_names[col].max() - reduced_with_names[col].min())
plt.figure(figsize=(7, 7), tight_layout=True)

# plot both the reduced and then selected and the selected and then reduced:
plt.scatter(df_reduced['Dim1'], df_reduced['Dim2'], alpha=0.5, label='Selected and Reduced')
plt.scatter(reduced_with_names['Dim1'], reduced_with_names['Dim2'], alpha=0.5, label='Reduced and Selected')
for i, txt in enumerate(reduced_with_names['name']):
    plt.annotate(txt, (reduced_with_names['Dim1'][i], reduced_with_names['Dim2'][i]), fontsize=8)
for i, txt in enumerate(df_reduced['name']):
    plt.annotate(txt, (df_reduced['Dim1'][i], df_reduced['Dim2'][i]), fontsize=8,horizontalalignment='center', verticalalignment='center')

buffer = 0.02  # Adjust the buffer distance as needed

for i, txt in enumerate(reduced_with_names['name']):
    if txt in df_reduced['name'].values:
        # Get coordinates for the two points
        x1 = df_reduced.loc[df_reduced['name'] == txt, 'Dim1'].values[0]
        y1 = df_reduced.loc[df_reduced['name'] == txt, 'Dim2'].values[0]
        x2 = reduced_with_names.loc[reduced_with_names['name'] == txt, 'Dim1'].values[0]
        y2 = reduced_with_names.loc[reduced_with_names['name'] == txt, 'Dim2'].values[0]

        # Compute the vector and its length
        dx = x2 - x1
        dy = y2 - y1
        length = np.sqrt(dx**2 + dy**2)

        if length == 0:
            continue  # Avoid division by zero if the points coincide

        # Compute the unit vector components
        ux = dx / length
        uy = dy / length

        # Adjust the start and end positions by the buffer
        x_start = x1 + buffer * ux
        y_start = y1 + buffer * uy
        x_end = x2 - buffer * ux
        y_end = y2 - buffer * uy

        # Draw the arrow with the adjusted coordinates
        plt.arrow(x_start, y_start, x_end - x_start, y_end - y_start,
                  head_width=0.02, head_length=0.02, fc='r', ec='r', length_includes_head=True)
plt.xlabel("PCA_1")
plt.ylabel("PCA_2")
plt.legend()

plt.title(f"{method} Dimensionality Reduction (n_components={n_components})")



In [ ]:
from adjustText import adjust_text
from pypalettes import load_cmap

load_cmap('ninetales')

def calculate_voronoi_homogeneity(df, resolution=300, metric='euclidean', plot=True):
    """
    Calculate Voronoi regions (based on the chosen distance metric)
    and compute a homogeneity parameter based on the area differences
    of the regions. Optionally display the Voronoi plot.

    Parameters:
        df (pd.DataFrame): DataFrame with 'Dim1', 'Dim2', and optionally 'name' columns.
        resolution (int): Number of grid points along each axis.
        metric (str): Distance metric to use, 'euclidean' or 'manhattan'.
        plot (bool): If True, plot the Voronoi regions over an extended grid.

    Returns:
        homogeneity (float): Coefficient of variation (std/mean) of the region areas.
        areas (dict): Dictionary mapping region index to its area (number of grid points).
    """
    df['Dim1'] /= df['Dim1'].max()
    df['Dim2'] /= df['Dim2'].max()
    # Create a grid over an extended space to clearly visualize boundaries.
    x = np.linspace(-0.05, 1.05, resolution)
    y = np.linspace(-0.05, 1.05, resolution)
    xx, yy = np.meshgrid(x, y)
    grid_points = np.column_stack([xx.ravel(), yy.ravel()])

    # Extract the data points from the DataFrame.
    points = df[['Dim1', 'Dim2']].values

    # Compute distances from every grid point to every data point.
    if metric == 'euclidean':
        # Squared Euclidean distance: no need for the square root for ranking
        dists = np.sum((grid_points[:, None, :] - points[None, :, :])**2, axis=2)
    elif metric == 'manhattan':
        dists = np.sum(np.abs(grid_points[:, None, :] - points[None, :, :]), axis=2)
    else:
        raise ValueError("Unsupported metric. Choose 'euclidean' or 'manhattan'.")

    # For each grid point, find the index (in df) of the closest data point.
    nearest = np.argmin(dists, axis=1)

    # Reshape the nearest indices to match the grid's shape.
    voronoi_map = nearest.reshape(xx.shape)

    # Calculate the area (count of grid points) for each region.
    areas = {}
    num_points = points.shape[0]
    for idx in range(num_points):
        areas[idx] = np.sum(voronoi_map == idx)

    # Compute a homogeneity parameter: coefficient of variation (std/mean).
    area_values = np.array(list(areas.values()))
    mean_area = np.mean(area_values)
    std_area = np.std(area_values)
    homogeneity = std_area / mean_area  # Lower means more homogeneous

    # If requested, plot the Voronoi regions.
    if plot:
        plt.figure(figsize=(8, 8))
        # Use pcolormesh to show the regions.
         # Use pcolormesh, but rasterize it so it doesn't bloat the SVG
        pcm = plt.pcolormesh(xx, yy, voronoi_map, cmap='ninetales', shading='auto', alpha=0.5)
        pcm.set_rasterized(True)  # Rasterize only this element
        plt.scatter(points[:, 0], points[:, 1], c='black', edgecolor='white', s=80, zorder=10)

        # Optionally annotate with names, if the 'name' column exists.
    # Create list for texts
        texts = []
        if 'paper_name' in df.columns:
            for idx, row in df.iterrows():
                # Create a text object and initially place it slightly offset from the point.
                text = plt.text(row['Dim1'], row['Dim2'], str(row['paper_name']),
                               color='black', ha='center', va='bottom', fontsize=12, zorder=20)
                texts.append(text)

        # Adjust texts to minimize overlapping
        adjust_text(texts, arrowprops=dict(arrowstyle='-', color='gray'))

        # plt.title(f"Voronoi Regions ({metric.capitalize()} distance)\nHomogeneity (CV): {homogeneity:.3f}")
        plt.xlabel("Dim1")
        plt.ylabel("Dim2")
        plt.xlim(-0.05, 1.05)
        plt.ylim(-0.05, 1.05)

        plt.savefig(f"homogeneity_{metric}.svg", dpi=400)
        plt.show()

    return homogeneity, areas

reduced_with_names = pd.read_csv('buchwald_ligand_umap.csv')

print(reduced_with_names)

# plot_voronoi_regions(df_reduced, resolution=1000)
calculate_voronoi_homogeneity(reduced_with_names[reduced_with_names['name'].isin(['XPhos', 'Ephos', 'BrettPhos', 'MorDalPhos', 'P(tBu)3', 'PCy3', 'SPhos'])], resolution=1000)
# calculate_voronoi_homogeneity(reduced_with_names, resolution=1000, metric='manhattan')


# Search for the best point fillers

In [ ]:
# get the subset from reduced all and remove from the original:
reduced_search_lower = pd.merge(df_reduced_all, smiles_with_descriptors ,left_on='canonical_smiles', right_on = 'canonical_smiles_x', how='inner')
# just keep name canonical smiles and dimensions:
reduced_search_lower = reduced_search_lower[['name', 'canonical_smiles_x', 'Dim1', 'Dim2']]
# rename the columns:
reduced_search_lower = reduced_search_lower.rename(columns={'canonical_smiles_x': 'canonical_smiles'})

print(reduced_search_lower.head())

# now remove these row from df_reduced_all
# remove the rows from df_reduced_all that are in reduced_search_lower
reduced_search_upper = df_reduced_all[~df_reduced_all['canonical_smiles'].isin(reduced_search_lower['canonical_smiles'])]
# keep only canonical smiles and the dimensions:
reduced_search_upper = reduced_search_upper[['canonical_smiles', 'Dim1', 'Dim2']]

print(reduced_search_upper.head())
# remove

In [ ]:
import itertools
import heapq
import math

def evaluate_combination(lower_df, upper_subset, resolution=300, metric='euclidean'):
    """
    Merge the lower and chosen upper rows, renormalize the coordinates to [0,1],
    and compute the homogeneity value.

    Parameters:
        lower_df (pd.DataFrame): DataFrame with the lower set.
        upper_subset (pd.DataFrame): DataFrame with 3 rows from the upper set.
        resolution (int): Resolution for the Voronoi calculation.
        metric (str): Distance metric to use ('euclidean' or 'manhattan').

    Returns:
        homogeneity (float): The computed homogeneity coefficient.
    """
    # Merge the two DataFrames.
    merged = pd.concat([lower_df, upper_subset], ignore_index=True)
    # Re-normalize Dim1 and Dim2 to [0,1] using min-max normalization.
    for dim in ['Dim1', 'Dim2']:
        merged[dim] = (merged[dim] - merged[dim].min()) / (merged[dim].max() - merged[dim].min())
    homo, _ = calculate_voronoi_homogeneity(merged, resolution=resolution, metric=metric, plot=False)
    return homo

def find_best_combinations(lower_df, upper_df, k=200, resolution=300, metric='euclidean', sample=False, sample_n=10000, n_comb = 3):
    """
    Find the best k combinations (of 3 items from upper_df) to add to lower_df such
    that the Voronoi homogeneity is minimized (i.e. regions are as equal as possible).

    Parameters:
        lower_df (pd.DataFrame): DataFrame with the lower set.
        upper_df (pd.DataFrame): DataFrame with the upper set.
        k (int): Number of best combinations to keep.
        resolution (int): Grid resolution for Voronoi homogeneity calculation.
        metric (str): Distance metric ('euclidean' or 'manhattan').
        sample (bool): If True, sample a subset of combinations instead of iterating exhaustively.
        sample_n (int): Number of combinations to sample (only used if sample=True).

    Returns:
        best_combos (list): A list of tuples (homogeneity, combination_indices, combination_df).
    """
    best_heap = []  # will store tuples as (-homogeneity, combination, combination_df)

    # Create an iterator for combinations of indices of upper_df.
    all_indices = upper_df.index.tolist()
    comb_iter = itertools.combinations(all_indices, n_comb)
    total_combinations = sample_n if sample else math.comb(len(all_indices), n_comb)
    print(f"Total combinations to evaluate: {total_combinations}")

    if sample:
        # If sampling, randomly choose a subset of combinations.
        # Note: For a rigorous random sample of combinations, convert iterator to list isn't feasible,
        # so use np.random.choice with replacement on a sufficiently large number.
        comb_iter = (tuple(np.random.choice(all_indices, 3, replace=False)) for _ in range(sample_n))

    count = 0
    for comb in comb_iter:
        # Extract the subset from upper_df.
        subset = upper_df.loc[list(comb)]
        homo = evaluate_combination(lower_df, subset, resolution=resolution, metric=metric)
        # For the heap, we use -homo so that the highest homo among the kept ones is at the top.
        entry = (-homo, comb, subset.copy())
        if len(best_heap) < k:
            heapq.heappush(best_heap, entry)
        else:
            # If the current worst best (i.e. max homo in our k best) is worse than the new one,
            # replace it. Note: best_heap[0] gives the current maximum (because of negative values).
            if -best_heap[0][0] > homo:
                heapq.heapreplace(best_heap, entry)
        count += 1
        if count % 1000 == 0:
            percent_comb = count/total_combinations*100
            print(f"Processed {count}/{total_combinations} combinations wich is {percent_comb:.4f}% of the total.", end='\r')

    # Sort the heap by actual homogeneity value (lowest first)
    best_combos = sorted([(-item[0], item[1], item[2]) for item in best_heap], key=lambda x: x[0])
    return best_combos

# for testing purposes get a 100 long subset of df_upper

reduced_search_lower = pd.read_csv('../ES127-FeaturizingLigands/reduced_search_lower.csv')
reduced_search_upper_minus = reduced_with_names[~reduced_with_names['name'].isin(reduced_search_lower['name'])]


best_results = find_best_combinations(reduced_search_lower, reduced_search_upper_minus, k = 200, resolution = 100, metric = 'euclidean')
print(best_results)

In [ ]:

# save the dfs:
reduced_search_lower.to_csv('reduced_search_lower.csv', index = False)
reduced_search_upper.to_csv('reduced_search_upper.csv', index = False)


In [ ]:
for i, (homogeneity, comb, comb_df) in enumerate(best_results):
    if i > 5:
        break
    print(f'Homogeneity: {homogeneity}, Combination_indice; {comb}, combination smiles:')
    print(comb_df['canonical_smiles'])
    # merge with lowe df:
    plot_df = pd.concat([reduced_search_lower, comb_df])
    for col in ['Dim1', 'Dim2']:
        plot_df[col] = (plot_df[col] - plot_df[col].min()) / (plot_df[col].max() - plot_df[col].min())


    calculate_voronoi_homogeneity(plot_df, plot=True, resolution=1000, metric='euclidean')

In [ ]:
# from functools import partial
# from multiprocessing import Pool, cpu_count
#
# # Define a worker that evaluates one combination.
#
# def evaluate_single_combination(comb, lower_df, upper_df, resolution, metric):
#     """
#     Given one combination (tuple of indices), compute its homogeneity when merged with lower_df.
#     """
#     subset = upper_df.loc[list(comb)]
#     homo = evaluate_combination(lower_df, subset, resolution=resolution, metric=metric)
#     return (homo, comb, subset.copy())
#
# def find_best_combinations_parallel(lower_df, upper_df, k=200, resolution=300,
#                                     metric='euclidean', sample=True, sample_n=10000,
#                                     processes=None):
#     """
#     Find the best k combinations (of 3 items from upper_df) to add to lower_df
#     such that the Voronoi homogeneity is minimized. This version uses parallel processing.
#
#     Parameters:
#         lower_df (pd.DataFrame): DataFrame with the lower set.
#         upper_df (pd.DataFrame): DataFrame with the upper set.
#         k (int): Number of best combinations to keep.
#         resolution (int): Grid resolution for Voronoi homogeneity calculation.
#         metric (str): Distance metric ('euclidean' or 'manhattan').
#         sample (bool): If True, sample a subset of combinations.
#         sample_n (int): Number of combinations to sample (if sample=True).
#         processes (int): Number of processes to use (default: cpu_count()).
#
#     Returns:
#         best_combos (list): A list of tuples (homogeneity, combination indices, combination_df).
#     """
#     if processes is None:
#         processes = cpu_count()
#
#     best_heap = []  # heap will store (-homogeneity, combination, combination_df)
#     all_indices = upper_df.index.tolist()
#
#     # Generate the combinations iterator.
#     comb_iter = itertools.combinations(all_indices, 3)
#
#     total_combinations = sample_n if sample else math.comb(len(all_indices), 3)
#
#     if sample:
#         comb_iter = (tuple(np.random.choice(all_indices, 3, replace=False)) for _ in range(sample_n))
#
#     # Create a partial function for the worker:
#     worker_func = partial(evaluate_single_combination, lower_df=lower_df,
#                           upper_df=upper_df, resolution=resolution, metric=metric)
#
#     count = 0
#
#     # Use a multiprocessing Pool to evaluate combinations in parallel.
#     with Pool(processes=processes) as pool:
#         # Use imap_unordered to get results as they come in.
#         for result in pool.imap_unordered(worker_func, comb_iter, chunksize=100):
#             homo, comb, subset_df = result
#             entry = (-homo, comb, subset_df)
#             if len(best_heap) < k:
#                 heapq.heappush(best_heap, entry)
#             else:
#                 if -best_heap[0][0] > homo:
#                     heapq.heapreplace(best_heap, entry)
#             count += 1
#             if count % 1000 == 0:
#                 percent = count / total_combinations * 100
#                 print(f"Processed {percent:.2f}% of combinations...", end='\r')
#
#     best_combos = sorted([(-item[0], item[1], item[2]) for item in best_heap], key=lambda x: x[0])
#     return best_combos
#
# best_results = find_best_combinations_parallel(reduced_search_lower, subset_df_upper, k = 200, resolution = 100, metric = 'euclidean')

reduced_with_names.to_csv("buchwald_ligand_umap.csv", index = False, columns = ['Dim1', 'Dim2', 'name', 'canonical_smiles'])

In [ ]:
# let's retry the calculation but let it pick sets of 10 or 12 ligands that make the space better


reduced_search_lower_minimal = pd.read_csv('../ES127-FeaturizingLigands/reduced_search_lower.csv')
reduced_search_lower_minimal = reduced_search_lower_minimal[reduced_search_lower_minimal['name'] == 'jerry']
reduced_search_upper_minus_minimal = reduced_with_names.copy()


best_results_minimal_set = find_best_combinations(reduced_search_lower_minimal, reduced_search_upper_minus_minimal, k = 200, resolution = 100, metric = 'euclidean',
                                      n_comb = 9)


In [ ]:
for i, (homogeneity, comb, comb_df) in enumerate(best_results_minimal_set):
    if i > 5:
        break
    print(f'Homogeneity: {homogeneity}, Combination_indice; {comb}, combination smiles:')
    print(comb_df['canonical_smiles'])
    # merge with lowe df:
    plot_df = pd.concat([comb_df])
    for col in ['Dim1', 'Dim2']:
        plot_df[col] = (plot_df[col] - plot_df[col].min()) / (plot_df[col].max() - plot_df[col].min())


    calculate_voronoi_homogeneity(plot_df, plot=True, resolution=1000, metric='euclidean')